In [12]:
import pandas as pd
import json
import os
import time

from openai import OpenAI
from tqdm import tqdm

## API 세팅

In [ ]:
SOLAR_API_KEY = ""
client = OpenAI(api_key=SOLAR_API_KEY, base_url="https://api.upstage.ai/v1")

## Few-Shot Prompt

In [14]:
SYSTEM_PROMPT = """너는 한국어 자연어 처리(NLP) 학습용 대화 요약 데이터셋을 구축하는 최고 수준의 AI 연구원이야.
내가 제시하는 [예시 데이터]의 말투, 대화 길이의 다양성(짧은 대화~긴 대화), 그리고 '요약문의 작성 규칙'을 완벽하게 분석하고 모방하여, 완전히 새로운 상황의 [주제]에 맞는 대화와 요약문 세트 5개를 창작해 줘.

[엄격한 작성 규칙]
1. 화자는 반드시 '#Person1#:', '#Person2#:' 형태로 표기할 것.
2. 대화문의 첫 줄은 무조건 '[Dialogue]\\n' 으로 시작하고, 대화 사이의 줄바꿈도 실제 엔터(Enter)가 아닌 이스케이프 문자('\\n')를 사용하여 완벽한 한 줄의 JSON 문자열(String) 규칙을 지킬 것.
3. 요약문은 철저하게 제3자 관점(예: "#Person1#은 #Person2#에게 ~한다고 말한다/합니다.")에서 객관적 팩트 위주로 1~2문장으로 압축할 것.
4. 출력 형식은 반드시 아래의 JSON 리스트 형식으로만 출력할 것. 다른 부연 설명은 절대 금지.
[
  {"topic": "새로운주제", "dialogue": "[Dialogue]\n#Person1#: ...", "summary": "..."},
  ...
]

[예시 데이터]
1. 주제: 생일 축하
대화: [Dialogue]\n#Person1#: 생일 축하해, Aims!\n#Person2#: 고마워, Lisa.\n#Person1#: 여기 선물이야. 마음에 들었으면 좋겠다.\n#Person2#: 정말 좋다! 이거 오랫동안 기다렸잖아.\n#Person1#: 그 말 들으니 정말 기쁘다.\n#Person2#: 이리 와봐, 친구들 좀 소개할게.
요약: Lisa가 Aims에게 생일 선물을 주었고, Aims는 그 선물을 매우 좋아합니다.

2. 주제: 고장 난 핸드폰 반품
대화: [Dialogue]\n#Person1#: 어떻게 도와드릴까요?\n#Person2#: 물건을 반품하고 싶어요.\n#Person1#: 어떤 걸 반품하시나요?\n#Person2#: 이 핸드폰을 반품하고 싶어요.\n#Person1#: 문제라도 있나요?\n#Person2#: 고장 났어요.\n#Person1#: 정확히 뭐가 문제인가요?\n#Person2#: 전화기가 자꾸 꺼져요.\n#Person1#: 알겠습니다. 영수증은 가지고 계신가요?\n#Person2#: 네, 여기 있습니다.\n#Person1#: 지금 바로 환불 처리해드릴게요.\n#Person2#: 정말 감사합니다.
요약: #Person2#가 고장으로 인해 자꾸 꺼지는 핸드폰을 반품하고, #Person1#이 환불을 처리해 줍니다.

3. 주제: 가족 문제
대화: [Dialogue]\n#Person1#: 이거 망가졌네! Jacky가 이걸 보면 안 좋아할 텐데. 이거 Jacky가 제일 좋아하는 CD야! 엄마한테 말할걸.\n#Person2#: 제발, Kathy, 나 10달러만 빌려줄 수 있어? 새 거 사줄게. 그리고 네 방도 청소해줄게.
요약: #Person2#가 Jacky의 CD를 망가뜨려 Kathy에게 새로 사기 위한 돈을 빌려달라고 부탁한다.
"""

## Data 생성

In [15]:
def generate_augmented_data(target_topic, num_samples=5):
    user_prompt = f"위 규칙과 예시를 바탕으로, '{target_topic}' 주제와 관련된 전혀 새로운 대화와 요약문 세트를 {num_samples}개 생성해 줘. 반드시 JSON 리스트로만 대답해."
    
    try:
        response = client.chat.completions.create(
            model="solar-pro", 
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7 # 너무 뻔하지 않게 약간의 창의성 부여
        )
        
        # 텍스트로 온 응답을 파이썬 리스트/딕셔너리로 변환
        result_text = response.choices[0].message.content.strip()
        
        # 마크다운 코드 블록(```json ... ```)이 섞여올 경우를 대비한 찌꺼기 제거
        if result_text.startswith("```"):
            result_text = result_text.strip("`").replace("json\n", "")
            
        json_data = json.loads(result_text, strict=False)
        return json_data
        
    except Exception as e:
        print(f"API 에러 또는 JSON 파싱 실패 : {e}")
        # print("원본 출력 결과:", response.choices[0].message.content) # 디버깅용
        return []

## Sample

In [16]:
test_results = generate_augmented_data(target_topic="컴퓨터 수리", num_samples=3)

print("\n🎯 [생성된 데이터 확인]")
for i, item in enumerate(test_results):
    print(f"\n--- 샘플 {i+1} ---")
    print(f"주제: {item['topic']}")
    print(f"대화:\n{item['dialogue']}")
    print(f"요약: {item['summary']}")


🎯 [생성된 데이터 확인]

--- 샘플 1 ---
주제: 컴퓨터 수리
대화:
[Dialogue]
#Person1#: 컴퓨터가 갑자기 꺼져서 안 켜져요. 어떻게 해야 할까요?
#Person2#: 전원 어댑터와 콘센트를 먼저 확인해주세요. 그 후 본체의 전원 버튼이 눌렸는지 보세요.
#Person1#: 다 확인했는데요. 어댑터도 새 거로 바꿔봤어요.
#Person2#: 그럼 내부 부품 문제일 수 있습니다. RAM이나 전원 공급 장치를 점검해보죠.
#Person1#: 직접 할 수 있을까요?
#Person2#: 기본 점검은 가능하지만, 전문 수리가 필요할 수도 있습니다. 일단 분해해서 먼지 제거부터 해보세요.
#Person1#: 알겠습니다. 시도해보겠습니다.
요약: #Person1#이 컴퓨터 전원 문제를 호소하자, #Person2#는 하드웨어 점검이 필요하다고 조언하며 직접 시도해볼 것을 권장합니다.

--- 샘플 2 ---
주제: 컴퓨터 수리
대화:
[Dialogue]
#Person1#: 모니터는 켜지는데 화면에 아무 것도 안 나와요. GPU 문제일까요?
#Person2#: 그래픽 카드와 모니터 연결 상태를 확인해주세요. HDMI 케이블이 제대로 꽂혔나요?
#Person1#: 네, 다른 케이블로 바꿔도 똑같아요.
#Person2#: 그럼 GPU가 제대로 장착되었는지, 다른 포트는 사용해보았나요?
#Person1#: DVI 포트로도 안 됩니다. BIOS에서도 인식 안 될 수 있나요?
#Person2#: 네, 그럴 수 있습니다. 다른 모니터로 테스트해보고, 안 되면 GPU를 교체하거나 A/S를 받아야 합니다.
#Person1#: 알겠습니다. 다른 모니터로 확인해보겠습니다.
요약: #Person1#이 모니터 출력 문제를 보고하자, #Person2#는 연결 상태와 GPU 점검이 필요하다고 안내하며 추가 확인 방법을 제시합니다.

--- 샘플 3 ---
주제: 컴퓨터 수리
대화:
[Dialogue]
#Person1#: 컴퓨터를 켰을 때 이상한 소리가 나고 팬이 너무 시끄

## Save

In [17]:
data_path = '../Data'
# 나중에 병합하기 위해 원본 데이터 로드
train_ori = pd.read_csv(os.path.join(data_path, 'train_processed.csv'))

In [18]:
# 1. 증강할 타겟 토픽 설정 (원하는 토픽으로 수정 하면서 써보기)
target_topics = [
    # 1. 협상 및 문제 해결 (숫자나 조건이 많이 나와서 요약이 까다로움)
    '가격 협상', '아파트 임대', '중고 거래', '환불 요청', '계약 취소',
    
    # 2. 전문/특수 상황 (특정 도메인 지식이 필요한 대화)
    '의료 상담', '법률 상담', '취업 면접', '보험 가입', '차량 수리',
    
    # 3. 여행/일정의 복합 상황 (여러 조건이 얽혀있는 대화)
    '호텔 체크인', '항공권 예약', '여행 계획', '비자 신청'
]
SAMPLES_PER_TOPIC = 10 # 토픽당 뽑을 개수
augmented_rows = []

In [19]:
# 2. API 호출 루프
for topic in tqdm(target_topics, desc="토픽별 증강 진행도"):
    new_data_list = generate_augmented_data(target_topic=topic, num_samples=SAMPLES_PER_TOPIC)
    
    if new_data_list:
        augmented_rows.extend(new_data_list)
        
    time.sleep(1) # API 속도 제한 방어

토픽별 증강 진행도: 100%|██████████| 14/14 [02:50<00:00, 12.17s/it]


In [20]:
# 3. JSON 리스트 -> Pandas 데이터프레임 변환
augmented_df = pd.DataFrame(augmented_rows)


# 4. 원본과 형식을 맞추기 위한 전처리 (fname 주입 및 컬럼 정렬)
augmented_df['fname'] = [f"aug_data_{i:04d}" for i in range(len(augmented_df))]
augmented_df = augmented_df[['fname', 'dialogue', 'summary', 'topic']]

In [21]:
# 저장1. 새로 증강한 데이터만 따로 저장
only_aug_path = os.path.join(data_path, 'train_augmented_only.csv')
augmented_df.to_csv(only_aug_path, index=False, encoding='utf-8-sig')
print(f"\n✅ [1/2] 증강 데이터 단독 저장 완료: {only_aug_path} (총 {len(augmented_df)}건)")


✅ [1/2] 증강 데이터 단독 저장 완료: ../Data/train_augmented_only.csv (총 140건)


In [22]:
# 저장2. 원본 + 증강 병합 데이터 별도 저장 (원본 훼손 방지)
final_train_df = pd.concat([train_ori, augmented_df], ignore_index=True)

# 모델 편향 방지를 위해 섞어주기 (Shuffle)
final_train_df = final_train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

merged_path = os.path.join(data_path, 'train_augmented_merged.csv')
final_train_df.to_csv(merged_path, index=False, encoding='utf-8-sig')
print(f"✅ [2/2] 원본+증강 병합 데이터 저장 완료: {merged_path} (총 {len(final_train_df)}건)")

✅ [2/2] 원본+증강 병합 데이터 저장 완료: ../Data/train_augmented_merged.csv (총 12597건)
